In [2]:
import pandas as pd
import numpy as np

# 1. Recreate the dataset
data = [
    (1, 169535646, '21/6/2019', 532, 'Truck'),
    (1, 734265621, '18/9/2030', 5733, 'Van'),
    (1, 747557690, '19/8/2027', 677, 'Car'),
    (1, 847557635, '18/6/2024', 234, 'NA'),
    (3, 697949195, '19/4/2021', 623, 'Van'),
    (3, 797969175, '19/4/2022', 907, 'Van'),
    (3, 797670195, '17/7/2022', 2314, 'NA'),
    (3, 872231801, '19/7/2026', 566, 'Car'),
    (4, 971694618, '19/1/2016', 788, 'Truck'),
    (4, 991694620, '1/7/2020', 4543, 'Van'),
    (10, 971696623, '28/5/2020', 432, 'NA'),
    (2, 166535746, '12/3/2027', 332, 'NA'),
    (2, 235265625, '21/1/2018', 733, 'NA'),
    (6, 275330646, '12/3/2027', 1332, 'NA'),
    (6, 635265643, '19/12/2022', 2475, 'NA'),
    (6, 215265656, '22/10/2012', 888, 'Van'),
    (11, 268535746, '21/7/2017', 5232, 'Truck'),
    (11, 334265625, '11/10/2028', 333, 'NA')
]

columns = ['client_id', 'Claim_id', 'claim_date', 'claim_amt', 'car_type']
df = pd.DataFrame(data, columns=columns)

# 2. Compute ClientStats (Replicating the SQL CTE)
# We use transform to apply the aggregation to the original rows without collapsing the dataframe
# transform('sum') formula: sum claim amt based on where criteria and repeat for all rows based on groupby (same client)
df['total_na_amt'] = df['claim_amt'].where(df['car_type'] == 'NA', 0).groupby(df['client_id']).transform('sum')
df['total_valid_amt'] = df['claim_amt'].where(df['car_type'] != 'NA', 0).groupby(df['client_id']).transform('sum')
df['valid_claim_count'] = (df['car_type'] != 'NA').astype(int).groupby(df['client_id']).transform('sum')

# 3. Apply the Allocation Logic
# To prevent division by zero warnings, we use np.where or replace 0 with NaN for the calculation
# Formula error when valid amt = 0 (client with only NA car type claim)
valid_amt_safe = df['total_valid_amt'].replace(0, np.nan) 


# 4.1. Default (Rule 3): Keep the original claim amount
df['Amt Allocation'] = df['claim_amt']

# 4.2 Create a mask for clients that actually need allocation
has_distribution = (df['valid_claim_count'] > 0) & (df['total_na_amt'] > 0)

# 4.3. rule 2: Set NA car type claim amt allocation to 0 (only for clients needing allocation)
df.loc[has_distribution & (df['car_type'] == 'NA'), 'Amt Allocation'] = 0

# 4.4. rule 1: Distribute to valid claims (only for clients needing allocation)
# Note: Since we filtered by valid_claim_count > 0, valid_amt_safe is guaranteed to be > 0, 
# so we don't even need the .fillna(0) anymore!
df.loc[has_distribution & (df['car_type'] != 'NA'), 'Amt Allocation'] = (
    df['claim_amt'] + (df['claim_amt'] / valid_amt_safe) * df['total_na_amt']
)

# 5. Round to 2 decimal places (Replicating CAST AS DECIMAL(18,2))
df['Amt Allocation'] = df['Amt Allocation'].round(2)

### Add this step if want to delete/drop the calculated columns ###
# df = df.drop(columns=['total_na_amt', 'total_valid_amt', 'valid_claim_count'])

# Sort by client and claim
df = df.sort_values(by=['client_id', 'Claim_id']).reset_index(drop=True)

# Optional: Format as currency strings with 2 decimal
df['claim_amt_fmt'] = df['claim_amt'].apply(lambda x: f"${x:,.2f}")
df['Amt Allocation_fmt'] = df['Amt Allocation'].apply(lambda x: f"${x:,.2f}")

# Print the FULL dataframe to verify the formula
print("\n--- Data for Verification ---")
display(df[['client_id', 'Claim_id', 'car_type', 'claim_amt', 'total_na_amt', 'total_valid_amt', 'valid_claim_count', 'Amt Allocation']])

# Print the result
display(df[['client_id', 'Claim_id', 'car_type', 'claim_amt_fmt', 'Amt Allocation_fmt']])


--- Data for Verification ---


,client_id,Claim_id,car_type,claim_amt,total_na_amt,total_valid_amt,valid_claim_count,Amt Allocation
0,1,169535646,Truck,532,234,6942,3,549.93
1,1,734265621,Van,5733,234,6942,3,5926.25
2,1,747557690,Car,677,234,6942,3,699.82
3,1,847557635,NA,234,234,6942,3,0.00
4,2,166535746,NA,332,1065,0,0,332.00
5,2,235265625,NA,733,1065,0,0,733.00
6,3,697949195,Van,623,2314,2096,3,1310.80
7,3,797670195,NA,2314,2314,2096,3,0.00
8,3,797969175,Van,907,2314,2096,3,1908.33
9,3,872231801,Car,566,2314,2096,3,1190.87


,client_id,Claim_id,car_type,claim_amt_fmt,Amt Allocation_fmt
0,1,169535646,Truck,$532.00,$549.93
1,1,734265621,Van,"$5,733.00","$5,926.25"
2,1,747557690,Car,$677.00,$699.82
3,1,847557635,NA,$234.00,$0.00
4,2,166535746,NA,$332.00,$332.00
5,2,235265625,NA,$733.00,$733.00
6,3,697949195,Van,$623.00,"$1,310.80"
7,3,797670195,NA,"$2,314.00",$0.00
8,3,797969175,Van,$907.00,"$1,908.33"
9,3,872231801,Car,$566.00,"$1,190.87"


The .loc function uses a comma to separate two things:

**has_distribution = (df['valid_claim_count'] > 0) & (df['total_na_amt'] > 0)**

**df.loc[has_distribution & (df['car_type'] == 'NA'), 'Amt Allocation'] = 0**

1. Before the comma: The Rows (the condition/filter)

2. After the comma: The Columns (where to apply the change) => Amt Allocation = 0 when meet condition